# 07 — Train E3 EfficientNet-B0 (Gate 5)
- **Model:** ImageNet-pretrained EfficientNet-B0 (torchvision `IMAGENET1K_V1`) with a new head Dropout(0.2) → Linear(1280 → 9); `src/models/efficientnet.py` (D022).
- **Stage A:** backbone frozen (its BatchNorm statistics too), head trained, lr 1e-3, max `training.epochs_transfer_head` epochs.
- **Stage B:** starts from the best stage-A checkpoint; the three last feature blocks (`features[6:]`) and the head are fine-tuned, lr 1e-4, max `training.epochs_finetune` epochs.
- **Selection:** early stopping and checkpoint selection on validation macro F1 in each stage; `E3_best.pt` is the better of the two stage-best checkpoints (D010).
- **The test set is not loaded in this notebook** (CONSTITUTION C3, D018).
- **Outputs (Drive):** `03_Checkpoints/E3/E3_stageA_best.pt`, `E3_stageA_last.pt`, `E3_stageB_best.pt`, `E3_stageB_last.pt`, `E3_best.pt`; `04_Results/experiments/E3/E3_history.csv` (with `stage`), `E3_curves.png`, `E3_val_per_class_metrics.csv`, `E3_val_confusion_matrix.png`, `E3_training_summary.json`.
- **If Colab disconnects:** reconnect and Run all; finished stages are skipped and the interrupted stage resumes. To start over, rename or delete `03_Checkpoints/E3/` first.
- **Runtime:** GPU (T4). The first epoch of a new session is slow (~16 min of cold Drive reads).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/KNV-DSEB/realwaste-classification.git"
REPO_DIR = Path("/content/realwaste-classification")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
# Forget src.* modules imported earlier in this session, so a re-run uses the code just pulled.
for module_name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[module_name]

print(subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout)

In [ ]:
import json
from datetime import datetime

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn

from src.dataset import get_dataloaders, load_class_mapping, load_frozen_split
from src.evaluate import classification_metrics, per_class_metrics, plot_confusion_matrix, predict
from src.models import build_model
from src.models.efficientnet import WEIGHTS
from src.train import build_optimizer, build_scheduler, fit, plot_history
from src.utils import count_parameters, environment_info, get_paths, load_config, set_seed

EXPERIMENT_ID = "E3"
cfg = load_config(REPO_DIR / "configs" / "config.yaml")
exp_cfg = cfg["experiments"][EXPERIMENT_ID]
assert WEIGHTS.name == exp_cfg["weights"], (WEIGHTS.name, exp_cfg["weights"])
MAX_EPOCHS = {"A": cfg["training"]["epochs_transfer_head"], "B": cfg["training"]["epochs_finetune"]}
paths = get_paths(cfg)
CHECKPOINT_DIR = paths.checkpoint_dir / EXPERIMENT_ID
RESULT_DIR = paths.result_dir / "experiments" / EXPERIMENT_ID
RESULT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then Run all.")
amp = exp_cfg["amp"] and device.type == "cuda"
env = environment_info()
print(json.dumps(env, indent=2))
print("device:", device, "| AMP:", amp, "| max epochs:", MAX_EPOCHS)

In [ ]:
split_df = load_frozen_split(cfg)  # SHA256-verified frozen grouped split (D016)
class_names = load_class_mapping(cfg)["class_name"].tolist()
num_classes = len(class_names)
train_loader, val_loader = get_dataloaders(cfg, split_df=split_df)  # train + validation only
print({s: int(n) for s, n in split_df["split"].value_counts().items()})
print("train batches:", len(train_loader), "| val batches:", len(val_loader))

In [ ]:
model_kwargs = {"num_classes": num_classes, "dropout": exp_cfg["dropout"]}
set_seed(cfg["project"]["seed"])  # identical new head on every fresh run
model = build_model(exp_cfg["model_name"], pretrained=True, **model_kwargs).to(device)
criterion = nn.CrossEntropyLoss()  # unweighted (D009)
total_parameters = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_parameters:,}")

In [ ]:
def run_stage(stage, run_info):
    stage_cfg = exp_cfg[f"stage_{stage.lower()}"]
    model.set_trainable_blocks(stage_cfg["trainable_blocks"])
    stage_exp_cfg = {**exp_cfg, "lr": stage_cfg["lr"]}
    optimizer = build_optimizer([p for p in model.parameters() if p.requires_grad], stage_exp_cfg)
    scheduler = build_scheduler(optimizer, stage_exp_cfg)
    print(f"Stage {stage}: {stage_cfg['trainable_blocks']} trainable feature blocks, "
          f"{count_parameters(model):,} trainable parameters, lr {stage_cfg['lr']}")
    return fit(
        model, train_loader, val_loader,
        optimizer=optimizer, scheduler=scheduler, criterion=criterion, device=device, amp=amp,
        num_classes=num_classes,
        max_epochs=MAX_EPOCHS[stage],
        patience=cfg["training"]["early_stopping_patience"],
        checkpoint_dir=CHECKPOINT_DIR,
        experiment_id=f"{EXPERIMENT_ID}_stage{stage}",
        history_path=RESULT_DIR / f"{EXPERIMENT_ID}_stage{stage}_history.csv",
        run_info={**run_info, "stage": stage},
    )


run_info = {
    "experiment_id": EXPERIMENT_ID,
    "run_id": f"{EXPERIMENT_ID}_{datetime.now():%Y%m%d-%H%M%S}",
    "model_name": exp_cfg["model_name"],
    "model_kwargs": model_kwargs,
    "weights": exp_cfg["weights"],
    "class_names": class_names,
    "split_sha256": cfg["split"]["split_sha256"],
    "git_commit": env["git_commit"],
}
result_a = run_stage("A", run_info)

In [ ]:
# Stage B starts from the best stage-A checkpoint (D010) and keeps stage A's run id.
model.load_state_dict(torch.load(result_a.best_path, map_location=device)["model"])
result_b = run_stage("B", result_a.run_info)

In [ ]:
# Final E3 checkpoint: the better of the two stage-best checkpoints (ties keep stage A).
selected_stage, selected = ("B", result_b) if result_b.best_val_macro_f1 > result_a.best_val_macro_f1 else ("A", result_a)
final = torch.load(selected.best_path, map_location=device)
final["run_info"] = {**final["run_info"], "selected_stage": selected_stage}
FINAL_PATH = CHECKPOINT_DIR / f"{EXPERIMENT_ID}_best.pt"
torch.save(final, FINAL_PATH)
print(f"Selected stage {selected_stage}, stage epoch {final['epoch']}: "
      f"validation macro-F1 {selected.best_val_macro_f1:.4f} -> {FINAL_PATH.name}")

stage_a_epochs = len(result_a.history)
history = pd.concat([
    result_a.history.assign(stage="A", stage_epoch=result_a.history["epoch"]),
    result_b.history.assign(stage="B", stage_epoch=result_b.history["epoch"],
                            epoch=result_b.history["epoch"] + stage_a_epochs),
], ignore_index=True)
history.to_csv(RESULT_DIR / f"{EXPERIMENT_ID}_history.csv", index=False)

fig = plot_history(history, f"{EXPERIMENT_ID} {exp_cfg['model_name']} - training curves (stage A | stage B)")
for ax in fig.axes:
    ax.axvline(stage_a_epochs + 0.5, color="grey", linestyle=":")
fig.savefig(RESULT_DIR / f"{EXPERIMENT_ID}_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Validation diagnostics of the selected checkpoint. The test set stays untouched.
model.load_state_dict(final["model"])
val = predict(model, val_loader, device, criterion, amp)
val_metrics = classification_metrics(val.y_true, val.y_pred, num_classes)
print("Validation:", {k: round(v, 4) for k, v in val_metrics.items()},
      "| macro-F1 recorded during training:", round(final["val_metrics"]["macro_f1"], 4))

val_per_class = per_class_metrics(val.y_true, val.y_pred, class_names)
val_per_class.to_csv(RESULT_DIR / f"{EXPERIMENT_ID}_val_per_class_metrics.csv", index=False)
print(val_per_class.to_string(index=False))

fig = plot_confusion_matrix(val.y_true, val.y_pred, class_names,
                            f"{EXPERIMENT_ID} {exp_cfg['model_name']} - validation confusion matrix")
fig.savefig(RESULT_DIR / f"{EXPERIMENT_ID}_val_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Run record (EXPERIMENT_PROTOCOL "Training records"; PROJECT_CONTROL_CENTER "Experiment Runs").
def stage_record(stage, result):
    stage_cfg = exp_cfg[f"stage_{stage.lower()}"]
    return {
        "trainable_blocks": stage_cfg["trainable_blocks"],
        "learning_rate": stage_cfg["lr"],
        "epochs_planned": MAX_EPOCHS[stage],
        "epochs_completed": int(result.history["epoch"].max()),
        "stopped_early": result.stopped_early,
        "best_epoch": result.best_epoch,
        "best_val_macro_f1": result.best_val_macro_f1,
        "training_time_minutes": round(float(result.history["seconds"].sum()) / 60, 1),
        "checkpoint_path": str(result.best_path),
    }


summary = {
    **{k: v for k, v in result_a.run_info.items() if k != "stage"},
    "seed": cfg["project"]["seed"],
    "batch_size": cfg["training"]["batch_size"],
    "optimizer": exp_cfg["optimizer"],
    "weight_decay": exp_cfg["weight_decay"],
    "scheduler": exp_cfg["scheduler"],
    "early_stopping": f"{cfg['training']['early_stopping_monitor']}, patience {cfg['training']['early_stopping_patience']}",
    "augmentation": "D011 training profile",
    "class_weighting": cfg["data"]["class_weighting"],
    "amp": amp,
    "stages": {"A": stage_record("A", result_a), "B": stage_record("B", result_b)},
    "selected_stage": selected_stage,
    "best_epoch": int(history.loc[(history["stage"] == selected_stage)
                                  & (history["stage_epoch"] == selected.best_epoch), "epoch"].iloc[0]),
    "selection_metric": cfg["training"]["selection_metric"],
    "best_val_macro_f1": selected.best_val_macro_f1,
    "best_val_metrics": val_metrics,
    "parameter_count": total_parameters,
    "training_time_minutes": round(float(history["seconds"].sum()) / 60, 1),
    "checkpoint_path": str(FINAL_PATH),
    "environment": env,
    "config_snapshot": cfg,
    "notes": "",
}
with open(RESULT_DIR / f"{EXPERIMENT_ID}_training_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps({k: v for k, v in summary.items() if k not in ("environment", "config_snapshot", "class_names")}, indent=2))

## Next
1. Save this notebook with outputs to GitHub (File → Save a copy in GitHub) and send the summary printed above, the training curves and the validation confusion matrix.
2. Copy the run into `PROJECT_CONTROL_CENTER.xlsx` → *Experiment Runs* and `experiments/E3_EfficientNet_B0/README.md`.
3. Do not evaluate on the test set here; that happens once for E1–E3 in the final-evaluation notebook.